# IE2026 Task 1b — Majority Vote Combiner (Runs 4 + 5a + 5b)

**No GPU required.** Combines predictions from three Latin square permutations via majority vote to cancel position bias.

| Run | Permutation | Statement order | File needed |
|---|---|---|---|
| Run 4 | A | [1, 2, 3] | `prediction_run4_answer_first_en.csv` |
| Run 5a | D | [2, 3, 1] | `prediction_run5a_perm_D_en.csv` |
| Run 5b | E | [3, 1, 2] | `prediction_run5b_perm_E_en.csv` |

Upload all three CSV files to this Kaggle notebook before running.

**How majority vote works:** for each item, each run votes for one statement (the one it marked True). The statement with 2 or 3 votes wins. In case of a 3-way tie (all three runs pick different statements), Run 4 (the strongest individual run) is used as tiebreaker.


## 1. Load the three prediction CSVs


In [1]:
import pandas as pd
import csv, zipfile, os

# ── File paths — adjust if your Kaggle input names differ ────────────────────
# Upload these via the Kaggle 'Add data' panel → 'Upload' → select the CSVs
FILES = {
    'run4':  '/kaggle/input/datasets/syedmohaiminulhoque/permutation-ensemble/prediction_run4_answer_first_en.csv',
    'run5a': '/kaggle/input/datasets/syedmohaiminulhoque/permutation-ensemble/prediction_run5a_perm_D_en.csv',
    'run5b': '/kaggle/input/datasets/syedmohaiminulhoque/permutation-ensemble/prediction_run5a_perm_E_en.csv',
}

dfs = {}
for run, path in FILES.items():
    # Try Kaggle input directory first, then current directory
    for base in ['/kaggle/input', '.', '/kaggle/working']:
        full = os.path.join(base, path)
        if os.path.exists(full):
            dfs[run] = pd.read_csv(full)
            print(f'{run}: loaded {full} ({len(dfs[run])} rows)')
            break
    else:
        # Try searching recursively under /kaggle/input
        found = False
        for root, dirs, fnames in os.walk('/kaggle/input'):
            if path in fnames:
                dfs[run] = pd.read_csv(os.path.join(root, path))
                print(f'{run}: loaded {os.path.join(root, path)} ({len(dfs[run])} rows)')
                found = True
                break
        if not found:
            print(f'ERROR: {run} file not found: {path}')
            print('Upload it via Add data → Upload in the Kaggle sidebar.')

assert len(dfs) == 3, f'Need all 3 files, got {len(dfs)}'
print('\nAll three prediction files loaded.')


run4: loaded /kaggle/input/datasets/syedmohaiminulhoque/permutation-ensemble/prediction_run4_answer_first_en.csv (1500 rows)
run5a: loaded /kaggle/input/datasets/syedmohaiminulhoque/permutation-ensemble/prediction_run5a_perm_D_en.csv (1500 rows)
run5b: loaded /kaggle/input/datasets/syedmohaiminulhoque/permutation-ensemble/prediction_run5a_perm_E_en.csv (1500 rows)

All three prediction files loaded.


## 2. Pivot to wide format — one row per item


In [2]:
def load_votes(df):
    """
    From a long-format CSV (id, statement_index, prediction),
    return a dict mapping id -> chosen_index (0-based original).
    """
    votes = {}
    for _, row in df.iterrows():
        if str(row['prediction']).lower() == 'true':
            votes[row['id']] = int(row['statement_index'])
    return votes

vote4  = load_votes(dfs['run4'])
vote5a = load_votes(dfs['run5a'])
vote5b = load_votes(dfs['run5b'])

ids = sorted(vote4.keys())
print(f'Items in Run 4:  {len(vote4)}')
print(f'Items in Run 5a: {len(vote5a)}')
print(f'Items in Run 5b: {len(vote5b)}')

missing_5a = set(vote4) - set(vote5a)
missing_5b = set(vote4) - set(vote5b)
if missing_5a: print(f'WARNING: {len(missing_5a)} items missing from Run 5a')
if missing_5b: print(f'WARNING: {len(missing_5b)} items missing from Run 5b')
if not missing_5a and not missing_5b: print('All item IDs match across all three runs. ✓')


Items in Run 4:  500
Items in Run 5a: 500
Items in Run 5b: 500
All item IDs match across all three runs. ✓


## 3. Majority vote


In [3]:
from collections import Counter

final_votes = {}
n_unanimous = n_majority = n_tiebreak = 0

for iid in ids:
    v4  = vote4.get(iid)
    v5a = vote5a.get(iid, v4)   # fallback to run4 if missing
    v5b = vote5b.get(iid, v4)

    counts = Counter([v4, v5a, v5b])
    winner, top_count = counts.most_common(1)[0]

    if top_count == 3:
        n_unanimous += 1
    elif top_count == 2:
        n_majority += 1
    else:
        # 3-way tie: all three runs disagree — use Run 4 as tiebreaker
        winner = v4
        n_tiebreak += 1

    final_votes[iid] = winner

print(f'Total items:          {len(ids)}')
print(f'Unanimous (3/3):      {n_unanimous}  ({n_unanimous/len(ids)*100:.1f}%)')
print(f'Majority (2/3):       {n_majority}  ({n_majority/len(ids)*100:.1f}%)')
print(f'3-way tie (→ Run 4):  {n_tiebreak}  ({n_tiebreak/len(ids)*100:.1f}%)')
print()
print('Majority vote decisions are the cases where permutation ensembling changed')
print('the prediction vs Run 4 alone. Compare n_majority against the 10% ordering-sensitive rate.')


Total items:          500
Unanimous (3/3):      449  (89.8%)
Majority (2/3):       49  (9.8%)
3-way tie (→ Run 4):  2  (0.4%)

Majority vote decisions are the cases where permutation ensembling changed
the prediction vs Run 4 alone. Compare n_majority against the 10% ordering-sensitive rate.


## 4. Score against dev labels (if available)


In [4]:
# To score locally: set SPLIT='dev' and upload the dev JSONL alongside the CSVs.
# The dev JSONL is at QCRI/AynVQA-ArabicNLP26 task1b/dev_en.jsonl on HuggingFace.
DEV_JSONL = 'dev_en.jsonl'   # upload this file to Kaggle if you want a local score

gold = {}
for base in ['/kaggle/input', '.', '/kaggle/working']:
    for root, dirs, fnames in os.walk(base):
        if DEV_JSONL in fnames:
            import json as _json
            for line in open(os.path.join(root, DEV_JSONL)):
                r = _json.loads(line)
                if 'labels' in r:
                    gold[r['id']] = r['labels'].index(True)
            print(f'Loaded {len(gold)} dev labels from {os.path.join(root, DEV_JSONL)}')
            break
    if gold: break

if gold:
    scored_ids = [iid for iid in ids if iid in gold]
    correct_ensemble = sum(1 for iid in scored_ids if final_votes[iid] == gold[iid])
    correct_run4     = sum(1 for iid in scored_ids if vote4.get(iid) == gold[iid])
    total = len(scored_ids)

    ci_ensemble = 1 - correct_ensemble / total
    ci_run4     = 1 - correct_run4 / total

    print(f'\nScored on {total} dev items:')
    print(f'  Run 4 alone:   CI={ci_run4:.4f}  Acc={correct_run4/total:.4f}')
    print(f'  Ensemble A+D+E: CI={ci_ensemble:.4f}  Acc={correct_ensemble/total:.4f}')
    delta = ci_ensemble - ci_run4
    direction = 'better ↓' if delta < 0 else 'worse ↑'
    print(f'  Delta: {delta:+.4f} ({direction})')
    print()

    # Which items did the ensemble fix vs break?
    fixed   = [iid for iid in scored_ids
               if vote4.get(iid) != gold[iid] and final_votes[iid] == gold[iid]]
    broken  = [iid for iid in scored_ids
               if vote4.get(iid) == gold[iid] and final_votes[iid] != gold[iid]]
    print(f'  Fixed by ensemble:   {len(fixed)}')
    print(f'  Broken by ensemble:  {len(broken)}')
    print(f'  Net change:          {len(fixed) - len(broken):+d} items')
else:
    print('No dev labels found — upload dev_en.jsonl to score locally.')
    print('The ensemble predictions will still be saved for Codabench submission.')


No dev labels found — upload dev_en.jsonl to score locally.
The ensemble predictions will still be saved for Codabench submission.


## 5. Save combined predictions and Codabench zip


In [5]:
OUT_CSV = 'prediction_en.csv'
with open(OUT_CSV, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['id', 'statement_index', 'prediction'])
    for iid in ids:
        chosen = final_votes[iid]
        for si in range(3):
            w.writerow([iid, si, 'true' if si == chosen else 'false'])
print(f'Wrote {OUT_CSV}: {len(ids)*3} rows')

ZIP_NAME = 'prediction_en.zip'
with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(OUT_CSV, 'prediction_en.csv')
print(f'Wrote {ZIP_NAME}  →  upload to Codabench 17051 (1b English)')


Wrote prediction_en.csv: 1500 rows
Wrote prediction_en.zip  →  upload to Codabench 17051 (1b English)
